# Sentinel-3A OLCI Extraction

This notebook demonstrates how to search for Sentinel-3A OLCI granules via **earthaccess** and extract ocean color bands using **satpy**.

Sentinel-3 products ship as `.zip` archives — the extractor handles automatic expansion.

> **Note:** The search API returns coarse footprint polygons. The actual data swath may not cover your entire AOI. If the extracted raster is all NaN, the grid cells fall outside the true satellite footprint — try a different AOI or time range.

| Property | Value |
|----------|-------|
| 📡 Sensor | Sentinel-3A OLCI |
| ⏱ Estimated time | ~10 min |
| 💾 Disk usage | ~2 GB |
| 🔐 Auth required | Earthdata 🔐 |

> 📖 New to AER? Read the [main README](../../README.md) for the full quickstart and API overview.

## Setup

Import the required libraries and configure the search parameters.

In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import shutil
import time

import geopandas as gpd

from aer.client import AerClient
from aer.interfaces import ExtractionProfile
from aer.search_earthaccess import earthaccess_download_wrapper

## Configuration & Earthdata Authentication

Sentinel-3A OLCI L1B EFR granules are hosted by NASA (via ESA redistribution) and require **Earthdata authentication**.

`earthaccess.login()` handles credential caching (`.netrc` or environment variables). If you have not configured credentials, run `earthaccess.login()` interactively once to store them.

`plugin_hints={'search_earthaccess': collections}` forces AER to use the `search_earthaccess` plugin, which wraps NASA's CMR API. Without the hint, AER would still resolve it automatically because `search_earthaccess` declares `S3A_OL_1_EFR` in its `supported_collections`.

> **Note:** The search API returns coarse footprint polygons. The actual data swath may not cover your entire AOI. If the extracted raster is all NaN, the grid cells fall outside the true satellite footprint — try a different AOI or time range.

## Configuration

Define the AOI, time range, and collection.

In [ ]:
# --- Configuration ---
DATE_START = datetime(2026, 4, 1, 0, 0, tzinfo=timezone.utc)
DATE_END = datetime(2026, 4, 5, 0, 0, tzinfo=timezone.utc)

# Load AOI — Buenos Aires province (well within the 2026-04-01 swath footprint)
# The swath for this date covers roughly lon -63° to -45°, lat -45° to -31°
geojson_path = Path("../data/buenos_aires.geojson")
gdf = gpd.read_file(geojson_path)
aoi = gdf.geometry.iloc[0]

# Sentinel-3A OLCI L1B EFR collection
collections = ["S3A_OL_1_EFR"]

# --- Client Setup ---
client = AerClient()

## ExtractionProfile in Detail

`ExtractionProfile` is an `@attrs.frozen` dataclass defined in `aer.interfaces.core`. It has four fields:

1. **`name`** — Arbitrary label for your own bookkeeping.
2. **`resolution`** — Target pixel size in metres. OLCI EFR native resolution is ~300 m; we extract at 300 m.
3. **`collection_variables_map`** — Dict mapping each collection to the list of bands/variables you want.
   Here we ask for `Oa04` (cyan, ~490 nm) from `S3A_OL_1_EFR`.
4. **`extra_params`** — Plugin-specific key/value pairs. We pass `reader='olci_l1b'` so the satpy-based extractor knows which reader to instantiate.

You can define multiple profiles in one run (e.g. one for ocean colour, one for land). Each profile produces its own set of tasks and output files.

## Search

Search for Sentinel-3 OLCI granules via **earthaccess**.

In [ ]:
print("Searching...", flush=True)
results = client.search(
    collections=collections,
    start_datetime=DATE_START,
    end_datetime=DATE_END,
    intersects=aoi,
    plugin_hints={"search_earthaccess": collections},
)
print(f"Found {len(results)} results", flush=True)
results.head()

## Extraction Parameters & ZIP Handling

`extract_params` is a flat dict forwarded directly to the extractor plugin.

Because Sentinel-3 data is **not** public S3, we must provide a `downloader`. `earthaccess_download_wrapper` is a thin wrapper around `earthaccess.download()` that conforms to AER's `Downloader` protocol: it accepts `(url, local_path)` and blocks until the file is written. The extractor calls it for every granule URL.

Sentinel-3 products are delivered as `.zip` archives. The `extract_satpy` plugin automatically expands them and discovers the `.SEN3` directory contents before passing filenames to satpy.

Common satpy-related keys:

- `padding` — Extra pixels added around each grid cell to avoid edge artefacts.
- `resampler` — Resampling method for the target grid (`nearest`, `bilinear`, `native`).
- `calibration` — Radiometric calibration (`radiance`, `reflectance`, `brightness_temperature`).
- `reader` — Satpy reader name (can also live in `extra_params` of the profile).

`max_batch_workers=4` runs four processes in parallel. Use `None` for sequential execution (lower memory).

## Prepare Extraction

Define extraction profiles and prepare tasks.

In [ ]:
# --- Prepare Extraction ---
uri = "extract_buenos_aires_sentinel3"

profiles = [
    ExtractionProfile(
        name="olci_rgb",
        resolution=300,
        collection_variables_map={"S3A_OL_1_EFR": ["Oa04"]},
        extra_params={"reader": "olci_l1b"},
    )
]

prepare_params = {"cells_per_chunk": 10}

tasks = client.prepare_for_extraction(
    results,
    target_aoi=aoi,
    uri=uri,
    profiles=profiles,
    target_grid_dist=256000,
    target_grid_overlap=False,
    prepare_params=prepare_params,
    plugin_hints={"extract_satpy": collections},
)

print(f"Prepared {len(tasks)} extraction tasks", flush=True)

## Extract

Run the extraction pipeline.

> Note: Sentinel-3 products are delivered as `.zip` archives. The `extract_satpy` plugin will automatically expand them and discover the `.SEN3` directory contents.

In [ ]:
# Clean output directory
uri_path = Path(uri)
if uri_path.exists():
    shutil.rmtree(uri_path)
uri_path.mkdir(parents=True)

print("Extracting...", flush=True)
start_time = time.time()

extract_params = {
    "downloader": earthaccess_download_wrapper,
    "calibration": "reflectance",
    "padding": 2,
    "resampler": "nearest",
}

results_df = client.extract_batches(
    tasks,
    extract_params=extract_params,
    plugin_hints={"extract_satpy": collections},
    max_batch_workers=4,
)

end_time = time.time()
print(f"Extraction took {end_time - start_time:.2f}s")
print(f"Extracted {len(results_df)} artifacts")

## Results

Inspect the extracted artifacts.

In [ ]:
head = results_df[["id", "collection", "grid_cell", "uri"]].head()
head